# 🛠️ CIDA SRE Swarm: Production 72h Autonomous Orchestration & Benchmarking

This notebook demonstrates the deployment and verification of the **CIDA-SRE Agent Swarm** on enterprise-grade infrastructure. It handles:
1. Secure cloning of private components via **GitHub Personal Access Token**.
2. Loading the frozen backbone **Qwen2.5-7B-Instruct** alongside the trained **Kairos-7B-CIDA-v1.5** reasoning adapters.
3. Running a side-by-side performance audit against a vanilla **Qwen2.5-7B** model on highly complex, cascading, and adversarial scenarios.

---

## 🔑 Step 1: Secure Repository Access & Environment Setup
We authenticate with GitHub to pull private tools and dependencies, then install requirements like `bitsandbytes` for 4-bit quantization.

In [ ]:
import os
import sys
import getpass

# Enter your GitHub Classic Token or Fine-Grained Token to download secure modules
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN") or getpass.getpass("🔑 Enter GITHUB_TOKEN: ")

# Setup environment and pull from private repo
if GITHUB_TOKEN:
    print("Token acquired. Configuring authentication...")
    !git clone https://{GITHUB_TOKEN}@github.com/Kairatzh/CIDA-plugin.git
    if 'CIDA-plugin' not in sys.path:
        sys.path.append('CIDA-plugin')
else:
    print("⚠️ No token provided. Proceeding with local cached files.")

# Install dependencies for 4-bit loading
!pip install -q transformers accelerate bitsandbytes>=0.46.1 peft huggingface_hub matplotlib torchdiffeq


## ☁️ Step 2: Download trained CIDA 1.5 weights & Qwen2.5-7B Backbone
We initialize the models. To prevent T4 or local GPU OutOfMemory errors, we configure the orchestrator to load the 7B backbone in **4-bit precision** using `bitsandbytes`, then load the specialized CIDA adapters.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from cida_plugin.orchestrator import KairosOrchestrator

BACKBONE_NAME = "Qwen/Qwen2.5-7B-Instruct"
CIDA_ADAPTER_REPO = "kirtzh/Kairos-7B-CIDA-v1.5"

print(f"[CIDA-OS] Loading frozen backbone: {BACKBONE_NAME} in 4-bit...")

tokenizer = AutoTokenizer.from_pretrained(BACKBONE_NAME)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4'
)

print(f"[CIDA-OS] Initializing KairosOrchestrator with CIDA 1.5 weights from: {CIDA_ADAPTER_REPO}...")
orchestrator = KairosOrchestrator(
    backbone_model_name=BACKBONE_NAME,
    cida_model_path_or_repo=CIDA_ADAPTER_REPO,
    device="cuda",
    quantization_config=bnb_config
)
print("✅ CIDA Orchestrator + Qwen2.5-7B in 4-bit successfully initialized.")


## 🧪 Step 3: Run SRE Scenarios (CIDA 1.5 vs Vanilla Qwen2.5-7B)
Now we launch the evaluation suite. The tasks here represent highly advanced production incidents:
- **Zero-Day Supply Chain Hijacks** (requires mapping logs to temporal metadata and rolling back deployments).
- **Adversarial Logic Traps** (CPU metrics spike, but database connections are starved; scaling replicas results in instant cascade failures).
- **Multi-day APT Backdoors** (requires memory lookbacks beyond normal LLM context windows to trace connection events 72 hours ago).

Let's execute the benchmark.


In [ ]:
# Reconfigure output for Windows UTF-8 compatibility
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

from cida_sre_demo.run_demo import run_evaluation
run_evaluation(
    backbone_model=orchestrator.llm,
    orchestrator=orchestrator,
    tokenizer=tokenizer,
    use_real_inference=True
)

## 📊 Step 4: Render Comparative Performance Analysis

In [ ]:
import json
import matplotlib.pyplot as plt

with open('cida_sre_demo/evaluation_results.json', 'r') as f:
    res = json.load(f)

agents = ['Vanilla Qwen2.5-7B', 'Qwen2.5-7B + CIDA 1.5']
solved = [res['base_llm']['solved'], res['cida']['solved']]
errors = [res['base_llm']['errors'], res['cida']['errors']]
costs = [res['base_llm']['cost_usd'], res['cida']['cost_usd']]

fig, axs = plt.subplots(1, 2, figsize=(15, 6))

# Chart 1: Scenario Resolution Rate
axs[0].bar(agents, solved, color=['#e74c3c', '#2ecc71'], width=0.5, edgecolor='black', linewidth=1)
axs[0].set_title('💡 Scenarios Solved (out of 30)', fontsize=12, fontweight='bold', pad=15)
axs[0].set_ylabel('Success Count', fontsize=10)
axs[0].set_ylim(0, 32)
axs[0].grid(axis='y', linestyle='--', alpha=0.5)
for idx, val in enumerate(solved):
    axs[0].text(idx, val + 0.6, f"{val}/30 ({round(val/30*100)}%)", ha='center', fontweight='bold', color='black')

# Chart 2: Cumulative Inference Cost
axs[1].bar(agents, costs, color=['#f39c12', '#3498db'], width=0.5, edgecolor='black', linewidth=1)
axs[1].set_title('💸 Total Inference Cost (USD)', fontsize=12, fontweight='bold', pad=15)
axs[1].set_ylabel('Cost in USD', fontsize=10)
axs[1].grid(axis='y', linestyle='--', alpha=0.5)
for idx, val in enumerate(costs):
    axs[1].text(idx, val + (val * 0.05), f"${val:.4f}", ha='center', fontweight='bold', color='black')

plt.suptitle('CIDA-SRE v1.5 vs Vanilla Qwen2.5-7B: 72-Hour SRE Evaluation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()